# ASTScribe — quick examples (run in Google Colab)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edujbarrios/astscribe/blob/main/examples/notebooks/quick_examples.ipynb)

**Understand what your ML notebook code does without leaving the notebook.**

Run this notebook top-to-bottom using **Runtime → Run all**. These examples demonstrate evidence-backed explanations, training checks, experiment reconstruction, diagnostics, and change impact. **ASTScribe analyzes Python source strings without executing the ML operations inside them.** You do not need PyTorch, a GPU, a downloaded model, or API keys.

The first cell installs ASTScribe automatically if needed. Afterwards, run each cell and compare its output with the saved results.

## 0 · Install ASTScribe

This cell is safe to run locally or in Colab. It installs ASTScribe only if it is absent, and prints one predictable confirmation.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("astscribe") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "astscribe"],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

from astscribe import NotebookAnalyzer, analyze, explain

print("ASTScribe ready")

ASTScribe ready


## 1 · Explain a model's inference procedure

**Problem:** You see a PyTorch code cell and want to understand what it does, not just whether it runs.

The source below is a string; ASTScribe **does not run the model**. The output is a readable explanation you can inspect while you work.

In [2]:
inference_source = """import torch
model.eval()
with torch.no_grad():
    outputs = model(inputs)
"""
print(explain(inference_source, style="scientific"))

Inference procedure

Gradient tracking is disabled for the enclosed operations, so no autograd graph is constructed for computations executed within this context.

The model is explicitly configured in evaluation mode.

A forward pass is performed by invoking the model on the supplied inputs.


## 2 · Check a training step and find the evidence

**Problem:** Does a training cell contain backward propagation and an optimizer update? Can you point to the source line behind the claim?

ASTScribe returns both an explanation and structured flags with supporting source locations.

In [3]:
training_source = """import torch
torch.manual_seed(42)
model.train()
optimizer.zero_grad()
outputs = model(inputs)
loss = criterion(outputs, targets)
loss.backward()
optimizer.step()
"""
result = analyze(training_source)
print(explain(training_source, style="concise"))
print("Backward pass:", result.training_step.backward_pass)
print("Parameter update:", result.training_step.parameter_update)
for claim in result.claims:
    if claim.rule == "pytorch.optimizer_step":
        print(f"Evidence: line {claim.line_start} ({claim.rule})")

Performs a PyTorch gradient-based training step with backward propagation and a parameter update.
Backward pass: True
Parameter update: True
Evidence: line 8 (pytorch.optimizer_step)


## 3 · Understand an unfamiliar experiment

**Problem:** Your colleague shares a multi-cell notebook built with Hugging Face. Which stages can you recognize before running its code?

ASTScribe statically reconstructs a pipeline from imports and operations; it **does not download** models or datasets.

In [4]:
workflow = NotebookAnalyzer.from_cells([
    """from datasets import load_dataset
train_data = load_dataset("imdb", split="train")""",
    """from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")""",
    """from transformers import Trainer, TrainingArguments
args = TrainingArguments(output_dir="runs", num_train_epochs=2)
trainer = Trainer(model=model, args=args, train_dataset=train_data)
trainer.train()""",
])
print(workflow.render_pipeline())

Dataset
    ↓
Model architecture
    ↓
Training procedure


## 4 · Detect notebook cell-order pitfalls

**Problem:** A notebook works on one person's machine but fails when run from the top. Could an earlier cell depend on a variable declared later?

ASTScribe detects *static* forward references and reports the original cell indices. This is an audit signal, not proof of a runtime failure.

In [5]:
out_of_order = NotebookAnalyzer.from_cells([
    "features = preprocess(raw_data)",
    "raw_data = load_data()",
    "predictions = predict(features)",
])
for warning in out_of_order.diagnostics().by_code("dependency.forward_reference"):
    print(f"Cell {warning.cell}: {warning.symbol} defined later in cell {warning.related_cell}")

Cell 0: raw_data defined later in cell 1


## 5 · Check what depends on a changed cell

**Problem:** If you edit preprocessing, which downstream cells might produce stale results?

ASTScribe follows source-level symbol dependencies to estimate the impact. It does not actually execute or schedule cells.

In [6]:
pipeline_cells = NotebookAnalyzer.from_cells([
    "raw = load_data()",
    "cleaned = normalize(raw)",
    "features = make_features(cleaned)",
    "model = fit(features)",
    "score = evaluate(model)",
])
impact = pipeline_cells.impact(1)
print("Revisit cells:", impact.affected_cells)
print("Cells needed first:", impact.required_ancestors)
print("Blast radius:", impact.blast_radius)

Revisit cells: (2, 3, 4)
Cells needed first: (0,)
Blast radius: 3


## Bonus · Explain cells without leaving Jupyter

ASTScribe also has IPython magics. Run the next cell to load the extension. Then create a **new code cell** with:

```python
%%scribe concise
model.eval()
with torch.no_grad():
    outputs = model(inputs)
```

The cell magic displays a Markdown explanation **without executing its body**.

To explain a prior executed input using preceding static context, run `%scribe N concise` where `N` is the number in Jupyter's `In[N]` history. IPython-specific commands form conservative context boundaries.

In [7]:
%load_ext astscribe.ipython

## Next steps

- Inspect your own saved notebook: `NotebookAnalyzer.from_ipynb("my_notebook.ipynb").render_methodology(include_evidence=True)`.
- Read [full documentation and outputs](https://github.com/edujbarrios/astscribe#see-what-astscribe-produces).
- See [more executed notebooks](https://github.com/edujbarrios/astscribe/tree/main/examples/notebooks).

**Reminder:** ASTScribe explains *supported static evidence*; it cannot know hidden kernel state, runtime values, accuracy or convergence.